# 17 v2 · Audit газового bundle внутри и вне окон сервисных работ (CPU, без обучения)

**v2 (Drive 46):** preflight панели против bundle; молчание только между двумя реальными показаниями;
окна по индексу строки; тревоги G2 и R1 на одной eligible-когорте; контрольные окна не пересекаются с основным.

**Accelerator = None**, интернет не нужен. Ничего не обучается и не выбирается; bundle не меняется.

Вопрос: насколько качество переданной модели G2 (прогноз пересечения газа 1 % в D+2) держится **вне** периодов,
когда датчики снимали на поверку / проверяли газовой смесью. Спецификация согласована с GPT (Drive 38–44):
- скорится **неизменённый** `gas_cross_v3_bundle` и правило R1 на **одних и тех же** строках;
- окна работ по данным: все газовые каналы объекта молчат ≥ 4 суток → окно [начало − 3; конец + 21]
  (**post-hoc**, использует будущее — только для исследования меток; чувствительность +14 / +30);
- **причинный** флаг на D: такое молчание закончилось в последние 21 сутки и датчики уже вернулись;
- **negative control**: те же окна, сдвинутые на +30 и +60 суток;
- метрики по стратам: counts, unknown, prevalence, PR-AUC и **lift** G2 и R1, Brier/ECE, бутстрэп по неделям ΔPR-AUC;
  тревоги — **одна общая** политика top-25/сутки с cooldown, разбитая по стратам;
- периоды 2025 H2 и 2026 H1 — **ретроспективный** audit (оба уже исследовались), не нетронутый тест.

Выход: `results_17_maintenance_audit.json`, `summary_17_maintenance_audit_ru.md` — только агрегаты.


## Как запустить

1. **Add Input** → выход ноутбука 14 **той же версии, на которой обучался 16** (не перезапуск с исправлениями
   same-second). Если панель другая, ноутбук остановится с `PanelMismatch` и покажет найденный хеш — не обходить.
2. **Add Input** → dataset с двумя файлами: `gas_cross_v3_bundle.zip` и `gas_cross_v3_bundle.zip.sha256`
   (ровно те, что переданы backend; SHA сверяется автоматически).
3. **Accelerator → None**, `MODE = "FULL"` → **Save Version → Save & Run All** (минуты).


In [ ]:
# ЕДИНСТВЕННАЯ ЯЧЕЙКА, КОТОРУЮ МОЖНО ИЗМЕНЯТЬ
MODE = "FULL"   # "FULL" или "SMOKE" (1/10 каналов, проверка запуска)
print("Режим:", MODE)


In [ ]:
import os, warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy, pandas, polars, sklearn, lightgbm, sys
for m in (numpy, pandas, polars, sklearn, lightgbm):
    print(f"{m.__name__:10s}", m.__version__)
print("python    ", sys.version.split()[0])


## Встроенный код

Runtime 15 и audit 17 — те же файлы, что в `ml/sensor_failure/`. Predictor и модель берутся из подключённого bundle.


In [ ]:
# Исходный код (репозиторий на Kaggle не нужен)
SOURCES = {
"target_models_v3_runtime.py": "\"\"\"Ноутбук 15: сравнение моделей на целях панели v3 (T2a/T2b/T2c/T1a/T4) и typed-score контракт для бэкенда.\n\nПротокол зафиксирован до обучения (docs/EXPERIMENT_15_TARGETS_V3_RU.md):\n- только pre-2025, фолды fold_2023 / fold_2024 как в ноутбуке 05; 2021 (переход системы мониторинга, FAQ)\n  и 30 суток прогрева после него исключены;\n- purging по d_label_decision_end: метка обучающей строки решена до начала калибровки, калибровочной — до валидации;\n- модели: P0 априорная частота по типу датчика, R1 правило (недавние эпизоды), L1 логистическая регрессия,\n  G1 LightGBM, G2 LightGBM без календаря, G3 LightGBM без типа датчика/системы (проверка shortcut);\n- метрики: PR-AUC, ROC-AUC, Brier/ECE после Platt (калибровка на H1), бюджетная кривая top-k в сутки\n  с cooldown 72 ч и без; парный блочный бутстрэп по неделям для G1 против лучшего простого baseline;\n- выбор и evidence_level по заранее заданным gates; результат — typed-score контракт без построчных прогнозов.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport time\nfrom datetime import date, timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nSCHEMA_VERSION = \"3.0\"\nTARGET_SPECS: dict[str, dict[str, Any]] = {\n    \"target_t2a_link_onset\": {\"tz\": \"T2a\", \"incident_type\": \"link_loss_onset\", \"family\": \"probability\",\n                              \"rule\": (\"f_link_onset_strict_30d\", \"f_link_onset_strict_7d\"), \"competing\": \"competing_link_onset_d1\"},\n    \"target_t2b_link_sustained\": {\"tz\": \"T2b\", \"incident_type\": \"link_loss_sustained\",\n                                  \"family\": \"conditional_proxy_score\",\n                                  \"rule\": (\"f_link_completed_long_30d\", \"f_link_onset_strict_7d\"), \"competing\": \"competing_link_onset_d1\"},\n    \"target_t2c_power_onset\": {\"tz\": \"T2c\", \"incident_type\": \"power_loss_onset\", \"family\": \"probability\",\n                               \"rule\": (\"f_power_onset_strict_30d\", \"f_power_onset_strict_7d\"), \"competing\": \"competing_power_onset_d1\"},\n    \"target_t2c_power_sustained\": {\"tz\": \"T2c\", \"incident_type\": \"power_loss_sustained\",\n                                   \"family\": \"conditional_proxy_score\",\n                                   \"rule\": (\"f_power_completed_long_30d\", \"f_power_onset_strict_7d\"), \"competing\": \"competing_power_onset_d1\"},\n    \"target_t1a_tech_value\": {\"tz\": \"T1a\", \"incident_type\": \"measurement_fault_onset\",\n                              \"family\": \"probability\",\n                              \"rule\": (\"f_n_tech_like_30d\", \"f_n_undefined_measurement_30d\"), \"competing\": None},\n    \"target_t4_gas_cross\": {\"tz\": \"T4\", \"incident_type\": \"gas_threshold_cross\", \"family\": \"probability\",\n                            \"rule\": (\"f_gas_max_7d\", \"f_gas_band_n_7d\"), \"competing\": \"competing_gas_cross_d1\"},\n}\nCATEGORICAL = [\"link_state\", \"power_state\", \"тип_датчика\", \"тип_инж_системы\"]\nCALENDAR = [\"d_weekday\", \"d_month\"]\nCATALOGUE = [\"тип_датчика\", \"тип_инж_системы\"]\nFORBIDDEN_PREFIXES = (\"target_\", \"reason_\", \"competing_\", \"d_label\", \"d_target\", \"d_channel\", \"d_object\", \"d_cutoff\")\nFOLDS = [\n    {\"name\": \"fold_2023\", \"train_end\": date(2023, 1, 1), \"calibration\": (date(2023, 1, 1), date(2023, 7, 1)),\n     \"validation\": (date(2023, 7, 1), date(2024, 1, 1))},\n    {\"name\": \"fold_2024\", \"train_end\": date(2024, 1, 1), \"calibration\": (date(2024, 1, 1), date(2024, 7, 1)),\n     \"validation\": (date(2024, 7, 1), date(2025, 1, 1))},\n]\nLOAD_END = date(2025, 1, 1)\nEXCLUDED = (date(2021, 1, 1), date(2022, 1, 31))   # 2021 + 30 суток прогрева окон признаков\n# selectable, но исход неизвестен: такие строки ранжируются в бюджете (слоты в unknown считаются), но не входят в PR-AUC\nUNKNOWN_REASONS = [\"unobserved_window\", \"ambiguous_onset\", \"censored_duration\", \"unknown_previous_reading\"]\nMODELS = [\"P0_type_prior\", \"R1_rule\", \"L1_logistic\", \"G1_lightgbm\", \"G2_no_calendar\", \"G3_no_catalogue\"]\nBASELINES_FOR_GATE = [\"P0_type_prior\", \"R1_rule\"]\n\n\ndef make_config_targets(mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:\n    if mode not in (\"FULL\", \"SMOKE\"):\n        raise ValueError(\"MODE должен быть FULL или SMOKE\")\n    smoke = mode == \"SMOKE\"\n    cfg = {\n        \"mode\": mode,\n        \"input_dir\": os.environ.get(\"LDT_KAGGLE_INPUT_DIR\", \"/kaggle/input\"),\n        \"output_dir\": os.environ.get(\"LDT_OUTPUT_DIR\", \"/kaggle/working\"),\n        \"targets\": list(TARGET_SPECS),\n        \"smoke_channel_share\": 10,\n        \"verify_sha256\": True,\n        \"random_seed\": 42,\n        \"lgbm_iterations\": 60 if smoke else 300,\n        \"max_neg_per_pos\": 30,\n        \"max_train_rows\": 300_000 if smoke else 2_000_000,\n        \"max_logistic_rows\": 100_000 if smoke else 500_000,\n        \"budgets_per_day\": [5, 10, 25, 50],\n        \"cooldown_days\": 3,\n        \"bootstrap_reps\": 20 if smoke else 100,\n        \"min_val_positives\": 30,\n        \"shortcut_tolerance\": 0.10,\n        \"ece_calibrated_max\": 0.02,      # probability_calibrated только при ECE ≤ 0,02 в обоих фолдах\n        \"min_known_share_selectable\": 0.5,  # g6: в валидации исход известен хотя бы у половины selectable строк\n        \"max_unknown_share_calibrated\": 0.10,  # probability_calibrated: unknown среди selectable ≤ 10 %\n        \"min_type_positives\": 30,        # срез по типам датчиков: типы с ≥ 30 положительными\n    }\n    cfg.update(overrides or {})\n    return cfg\n\n\n# ---------------------------------------------------------------------------\n# Данные\n# ---------------------------------------------------------------------------\ndef file_sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        while block := f.read(1 << 22):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef discover_panel(input_dir: Path, verify: bool) -> tuple[list[Path], dict[str, Any]]:\n    found = sorted(Path(input_dir).rglob(\"panel_manifest_v3.json\"))\n    if not found:\n        raise FileNotFoundError(\"panel_manifest_v3.json не найден: подключите выход ноутбука 14 как Input\")\n    manifest_path = found[0]\n    manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n    if manifest.get(\"schema_version\") != SCHEMA_VERSION:\n        raise ValueError(f\"ожидалась схема {SCHEMA_VERSION}, в manifest {manifest.get('schema_version')}\")\n    parts = []\n    for part in manifest[\"parts\"]:\n        p = manifest_path.parent / part[\"file\"]\n        if verify and file_sha256(p) != part[\"sha256\"]:\n            raise ValueError(f\"SHA-256 части {part['file']} не совпадает с manifest\")\n        parts.append(p)\n    import polars as pl\n\n    rows = sum(int(pl.scan_parquet(p).select(pl.len()).collect().item()) for p in parts)\n    if rows != manifest[\"rows\"]:\n        raise ValueError(f\"строк в частях {rows:,}, в manifest {manifest['rows']:,}\")\n    return parts, manifest\n\n\ndef safe_features(manifest: dict[str, Any], columns: list[str]) -> list[str]:\n    # тип датчика/системы — справочные метаданные канала (есть в панели, но не в списке признаков manifest v3)\n    feats = [c for c in dict.fromkeys(list(manifest[\"feature_columns\"]) + CATALOGUE) if c in columns]\n    bad = [c for c in feats if c.startswith(FORBIDDEN_PREFIXES)]\n    if bad:\n        raise ValueError(f\"запрещённые признаки: {bad}\")\n    return feats\n\n\ndef load_target(pl: Any, parts: list[Path], target: str, features: list[str], cfg: dict[str, Any],\n                load_end: date = LOAD_END) -> Any:\n    \"\"\"load_end: граница загрузки (15 — всегда 2025-01-01; финальный экспорт 16 задаёт свою по рецепту).\"\"\"\n    reason = \"reason_\" + target[len(\"target_\"):]\n    lf = pl.scan_parquet(parts)\n    lf = lf.filter((pl.col(\"d_cutoff_date\") < load_end) & (pl.col(\"d_label_decision_end\") <= load_end)\n                   & ~pl.col(\"d_cutoff_date\").is_between(EXCLUDED[0], EXCLUDED[1], closed=\"left\"))\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    numeric = [c for c in features if c not in CATEGORICAL]\n    comp = TARGET_SPECS[target].get(\"competing\")\n    extra = [comp] if comp and comp in lf.collect_schema().names() else []\n    cols = [\"d_channel_key\", \"d_cutoff_date\", \"d_label_decision_end\", target, reason] + extra + features\n    return (lf.filter(pl.col(target).is_not_null() | pl.col(reason).is_in(UNKNOWN_REASONS)).select(cols)\n            .with_columns([pl.col(c).cast(pl.Float32) for c in numeric]).collect())\n\n\nSTRESS_PERIOD = (date(2021, 1, 1), date(2022, 1, 1))   # переход системы мониторинга (FAQ): только stress-срез\n\n\ndef load_stress(pl: Any, parts: list[Path], target: str, features: list[str], cfg: dict[str, Any]) -> Any:\n    \"\"\"2021 — не участвует ни в обучении, ни в выборе; оценивается один раз моделями fold_2024 (GPT 20).\"\"\"\n    lf = pl.scan_parquet(parts).filter(pl.col(\"d_cutoff_date\").is_between(*STRESS_PERIOD, closed=\"left\")\n                                       & (pl.col(\"d_label_decision_end\") <= STRESS_PERIOD[1]) & pl.col(target).is_not_null())\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    numeric = [c for c in features if c not in CATEGORICAL]\n    return lf.select([target] + features).with_columns([pl.col(c).cast(pl.Float32) for c in numeric]).collect()\n\n\ndef coverage(pl: Any, parts: list[Path], targets: list[str], cfg: dict[str, Any]) -> dict[str, Any]:\n    lf = pl.scan_parquet(parts).filter((pl.col(\"d_cutoff_date\") < LOAD_END)\n                                       & ~pl.col(\"d_cutoff_date\").is_between(EXCLUDED[0], EXCLUDED[1], closed=\"left\"))\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    total = int(lf.select(pl.len()).collect().item())\n    out: dict[str, Any] = {\"rows_pre2025_without_2021\": total, \"targets\": {}}\n    for t in targets:\n        r = \"reason_\" + t[len(\"target_\"):]\n        g = lf.group_by(r).agg(pl.len()).collect()\n        reasons = {str(k): int(v) for k, v in g.iter_rows()}\n        labelled = reasons.get(\"positive\", 0) + reasons.get(\"negative\", 0) + reasons.get(\"negative_transient_only\", 0)\n        by_type = (lf.group_by(\"тип_датчика\").agg(pl.col(t).is_not_null().mean().alias(\"share\"), pl.len().alias(\"rows\"))\n                   .filter(pl.col(\"rows\") >= 1000).sort(\"rows\", descending=True).head(15).collect())\n        out[\"targets\"][t] = {\"labelled_rows\": labelled, \"labelled_share\": labelled / max(total, 1),\n                             \"reasons\": reasons,\n                             \"labelled_share_by_sensor_type\": {str(k or \"нет в справочнике\"): round(float(s), 4)\n                                                                for k, s, _ in by_type.iter_rows()}}\n    return out\n\n\ndef split_fold(pl: Any, frame: Any, fold: dict[str, Any]) -> tuple[Any, Any, Any]:\n    c0, c1 = fold[\"calibration\"]\n    v0, v1 = fold[\"validation\"]\n    d, e = pl.col(\"d_cutoff_date\"), pl.col(\"d_label_decision_end\")\n    train = frame.filter((d < fold[\"train_end\"]) & (e <= c0))\n    cal = frame.filter((d >= c0) & (d < c1) & (e <= v0))\n    val = frame.filter((d >= v0) & (d < v1) & (e <= v1))\n    return train, cal, val\n\n\ndef sample_train(pl: Any, train: Any, target: str, cfg: dict[str, Any]) -> Any:\n    pos = train.filter(pl.col(target) == 1)\n    neg = train.filter(pl.col(target) == 0)\n    n_neg = min(neg.height, max(pos.height * cfg[\"max_neg_per_pos\"], 1), max(cfg[\"max_train_rows\"] - pos.height, 1))\n    rate = n_neg / max(neg.height, 1)\n    neg = neg.sample(n=n_neg, seed=cfg[\"random_seed\"]) if n_neg < neg.height else neg\n    return pl.concat([pos.with_columns(pl.lit(1.0).alias(\"_w\")),\n                      neg.with_columns(pl.lit(1.0 / max(rate, 1e-9)).alias(\"_w\"))])\n\n\n# ---------------------------------------------------------------------------\n# Модели\n# ---------------------------------------------------------------------------\ndef _pandas(frame: Any, cols: list[str], levels: dict[str, list[str]]) -> Any:\n    import pandas as pd\n\n    x = pd.DataFrame({c: frame[c].to_numpy() for c in cols})\n    for c in cols:\n        if c in CATEGORICAL:\n            v = frame[c].fill_null(\"__NA__\").cast(str).to_numpy()\n            x[c] = pd.Categorical(v, categories=levels[c])\n    return x\n\n\nclass TypePrior:\n    def fit(self, train: Any, target: str, m: float = 50.0) -> \"TypePrior\":\n        import polars as pl\n\n        w = train[\"_w\"]\n        y = train[target].cast(float)\n        self.global_rate = float((w * y).sum() / w.sum())\n        g = train.with_columns(y.alias(\"_y\")).group_by(\"тип_датчика\").agg(\n            (pl.col(\"_w\") * pl.col(\"_y\")).sum().alias(\"p\"), pl.col(\"_w\").sum().alias(\"n\"))\n        self.rates = {k: (p + m * self.global_rate) / (n + m) for k, p, n in g.iter_rows()}\n        return self\n\n    def score(self, frame: Any) -> np.ndarray:\n        return np.array([self.rates.get(k, self.global_rate) for k in frame[\"тип_датчика\"].to_list()], dtype=float)\n\n\ndef rule_score(frame: Any, spec: dict[str, Any]) -> np.ndarray:\n    a, b = spec[\"rule\"]\n    s = np.zeros(frame.height)\n    for col, w in ((a, 1.0), (b, 0.5)):\n        if col in frame.columns:\n            s = s + w * np.nan_to_num(frame[col].to_numpy().astype(float))\n    return s\n\n\nclass Logistic:\n    def fit(self, train: Any, target: str, features: list[str], cfg: dict[str, Any]) -> \"Logistic\":\n        from sklearn.linear_model import LogisticRegression\n\n        if train.height > cfg[\"max_logistic_rows\"]:\n            train = train.sample(n=cfg[\"max_logistic_rows\"], seed=cfg[\"random_seed\"])\n        self.num = [c for c in features if c not in CATEGORICAL]\n        self.cat = {c: [str(v) for v in train[c].fill_null(\"__NA__\").cast(str).value_counts(sort=True)[c].head(50).to_list()]\n                    for c in features if c in CATEGORICAL}\n        x = self._x(train, fit=True)\n        self.model = LogisticRegression(C=1.0, max_iter=300, random_state=cfg[\"random_seed\"])\n        self.model.fit(x, train[target].to_numpy(), sample_weight=train[\"_w\"].to_numpy())\n        return self\n\n    def _x(self, frame: Any, fit: bool = False) -> np.ndarray:\n        num = np.column_stack([frame[c].to_numpy().astype(float) for c in self.num]) if self.num else np.zeros((frame.height, 0))\n        num = np.nan_to_num(num, nan=0.0, posinf=0.0, neginf=0.0)\n        num = np.sign(num) * np.log1p(np.abs(num))\n        if fit:\n            self.mu, self.sd = num.mean(0), num.std(0) + 1e-9\n        num = (num - self.mu) / self.sd\n        cats = [(frame[c].fill_null(\"__NA__\").cast(str).to_numpy()[:, None] == np.array(lv)[None, :]).astype(float)\n                for c, lv in self.cat.items()]\n        return np.hstack([num] + cats)\n\n    def score(self, frame: Any) -> np.ndarray:\n        return self.model.predict_proba(self._x(frame))[:, 1]\n\n\nclass LightGBM:\n    def fit(self, train: Any, target: str, features: list[str], cfg: dict[str, Any]) -> \"LightGBM\":\n        from lightgbm import LGBMClassifier\n\n        self.features = features\n        self.levels = {c: sorted(str(v) for v in train[c].fill_null(\"__NA__\").cast(str).unique().to_list())\n                       for c in features if c in CATEGORICAL}\n        self.model = LGBMClassifier(objective=\"binary\", n_estimators=int(cfg[\"lgbm_iterations\"]), learning_rate=0.05,\n                                    num_leaves=63, max_bin=63, min_child_samples=100, subsample=0.8, subsample_freq=1,\n                                    colsample_bytree=0.8, random_state=cfg[\"random_seed\"], n_jobs=-1, verbosity=-1)\n        cats = [c for c in features if c in CATEGORICAL]\n        self.model.fit(_pandas(train, features, self.levels), train[target].to_numpy(),\n                       sample_weight=train[\"_w\"].to_numpy(), categorical_feature=cats or \"auto\")\n        return self\n\n    def score(self, frame: Any) -> np.ndarray:\n        return self.model.predict_proba(_pandas(frame, self.features, self.levels))[:, 1]\n\n    def importance(self, top: int = 12) -> dict[str, float]:\n        gain = self.model.booster_.feature_importance(\"gain\")\n        total = float(gain.sum()) or 1.0\n        order = np.argsort(-gain)[:top]\n        return {self.features[i]: round(float(gain[i] / total), 4) for i in order}\n\n\n# ---------------------------------------------------------------------------\n# Метрики\n# ---------------------------------------------------------------------------\ndef _z(score: np.ndarray, is_probability: bool) -> np.ndarray:\n    if is_probability:\n        p = np.clip(score, 1e-6, 1 - 1e-6)\n        return np.log(p / (1 - p))\n    return np.log1p(np.maximum(score, 0))\n\n\ndef platt(raw_cal: np.ndarray, y_cal: np.ndarray, is_probability: bool) -> Any:\n    from sklearn.linear_model import LogisticRegression\n\n    if np.unique(y_cal).size < 2:\n        return None\n    m = LogisticRegression(max_iter=300)\n    m.fit(_z(raw_cal, is_probability).reshape(-1, 1), y_cal)\n    return m\n\n\ndef apply_platt(m: Any, raw: np.ndarray, is_probability: bool, prior: float) -> np.ndarray:\n    if m is None:\n        return np.full(len(raw), prior)\n    return m.predict_proba(_z(raw, is_probability).reshape(-1, 1))[:, 1]\n\n\ndef ece(y: np.ndarray, p: np.ndarray, bins: int = 10) -> float:\n    q = np.quantile(p, np.linspace(0, 1, bins + 1))\n    idx = np.clip(np.searchsorted(q, p, side=\"right\") - 1, 0, bins - 1)\n    return float(sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean()) for b in range(bins) if (idx == b).any()))\n\n\ndef budget_curve(days: np.ndarray, channels: np.ndarray, y: np.ndarray, score: np.ndarray,\n                 budgets: list[int], cooldown_days: int) -> dict[str, Any]:\n    \"\"\"top-k строк в сутки по всей selectable когорте; y: 1/0, −1 = исход неизвестен.\n    precision — нижняя граница (unknown = 0), precision_upper — верхняя (unknown = 1); recall — по известным положительным.\n    С cooldown канал не выдаётся повторно cooldown_days суток после алерта.\"\"\"\n    order = np.lexsort((-score, days))\n    d_sorted, c_sorted, y_sorted = days[order], channels[order], y[order]\n    starts = np.flatnonzero(np.r_[True, d_sorted[1:] != d_sorted[:-1]])\n    ends = np.r_[starts[1:], len(order)]\n    total_pos = int((y == 1).sum())\n    total_unk = int((y == -1).sum())\n    pos_sorted, unk_sorted = (y_sorted == 1).astype(int), (y_sorted == -1).astype(int)\n    out = {}\n    for k in budgets:\n        tp = n = unk = 0\n        tp_cd = n_cd = unk_cd = 0\n        last: dict[Any, int] = {}\n        for s, e in zip(starts, ends):\n            tp += int(pos_sorted[s:min(e, s + k)].sum())\n            unk += int(unk_sorted[s:min(e, s + k)].sum())\n            n += min(e - s, k)\n            day = int(d_sorted[s])\n            taken = 0\n            for i in range(s, e):\n                if taken >= k:\n                    break\n                ch = c_sorted[i]\n                if ch in last and day - last[ch] < cooldown_days:\n                    continue\n                last[ch] = day\n                taken += 1\n                tp_cd += int(pos_sorted[i])\n                unk_cd += int(unk_sorted[i])\n            n_cd += taken\n        out[str(k)] = {\"precision\": tp / max(n, 1), \"precision_upper\": (tp + unk) / max(n, 1), \"unknown_alerts\": unk,\n                       \"recall\": tp / max(total_pos, 1), \"alerts\": n,\n                       # recall по GPT 16: нижняя — все неизвестные вне бюджета положительные, верхняя — все выданные unknown положительные\n                       \"recall_low\": tp / max(total_pos + total_unk - unk, 1), \"recall_up\": (tp + unk) / max(total_pos + unk, 1),\n                       \"precision_cooldown\": tp_cd / max(n_cd, 1), \"precision_upper_cooldown\": (tp_cd + unk_cd) / max(n_cd, 1),\n                       \"unknown_alerts_cooldown\": unk_cd, \"recall_cooldown\": tp_cd / max(total_pos, 1),\n                       \"recall_low_cooldown\": tp_cd / max(total_pos + total_unk - unk_cd, 1),\n                       \"recall_up_cooldown\": (tp_cd + unk_cd) / max(total_pos + unk_cd, 1),\n                       \"alerts_cooldown\": n_cd}\n    return out\n\n\ndef evaluate(y: np.ndarray, raw: np.ndarray, cal: np.ndarray, budget_frame: tuple[np.ndarray, ...],\n             cfg: dict[str, Any]) -> dict[str, Any]:\n    \"\"\"Ранжирующие метрики и калибровка — по строкам с известным исходом; бюджет — по всей selectable когорте.\"\"\"\n    days, channels, y_all, raw_all = budget_frame\n    from sklearn.metrics import average_precision_score, roc_auc_score\n\n    ok = np.unique(y).size == 2\n    return {\n        \"pr_auc\": float(average_precision_score(y, raw)) if ok else None,\n        \"roc_auc\": float(roc_auc_score(y, raw)) if ok else None,\n        \"brier\": float(np.mean((cal - y) ** 2)),\n        \"ece10\": ece(y, cal),\n        \"budget\": budget_curve(days, channels, y_all, raw_all, cfg[\"budgets_per_day\"], cfg[\"cooldown_days\"]),\n    }\n\n\ndef by_type_pr_auc(y: np.ndarray, raw: np.ndarray, types: np.ndarray, min_pos: int) -> dict[str, float]:\n    from sklearn.metrics import average_precision_score\n\n    out = {}\n    for t in np.unique(types):\n        m = types == t\n        if y[m].sum() >= min_pos and y[m].min() != y[m].max():\n            out[str(t)] = float(average_precision_score(y[m], raw[m]))\n    return out\n\n\ndef paired_bootstrap(y: np.ndarray, a: np.ndarray, b: np.ndarray, weeks: np.ndarray, reps: int, seed: int) -> dict[str, Any]:\n    \"\"\"PR-AUC(a) − PR-AUC(b), блоки = календарные недели, одни и те же выборки для обеих моделей.\"\"\"\n    from sklearn.metrics import average_precision_score\n\n    uniq, inv = np.unique(weeks, return_inverse=True)\n    rng = np.random.default_rng(seed)\n    diffs = []\n    for _ in range(reps):\n        w = np.bincount(rng.integers(0, len(uniq), len(uniq)), minlength=len(uniq))[inv].astype(float)\n        m = w > 0\n        if y[m].min() == y[m].max():\n            continue\n        diffs.append(average_precision_score(y[m], a[m], sample_weight=w[m])\n                     - average_precision_score(y[m], b[m], sample_weight=w[m]))\n    d = np.array(diffs)\n    if not len(d):\n        return {\"mean\": None, \"ci95_low\": None, \"ci95_high\": None, \"reps\": 0}\n    return {\"mean\": float(d.mean()), \"ci95_low\": float(np.percentile(d, 2.5)),\n            \"ci95_high\": float(np.percentile(d, 97.5)), \"reps\": int(len(d))}\n\n\n# ---------------------------------------------------------------------------\n# Прогон\n# ---------------------------------------------------------------------------\ndef run_target(pl: Any, frame: Any, target: str, features: list[str], cfg: dict[str, Any], log: Any,\n               stress: Any = None) -> dict[str, Any]:\n    spec = TARGET_SPECS[target]\n    res: dict[str, Any] = {\"folds\": {}}\n    for fold in FOLDS:\n        train, cal, val_all = split_fold(pl, frame, fold)\n        known = pl.col(target).is_not_null()\n        train, cal, val = train.filter(known), cal.filter(known), val_all.filter(known)\n        n = {\"train\": train.height, \"calibration\": cal.height, \"validation\": val.height,\n             \"train_pos\": int(train[target].sum()), \"calibration_pos\": int(cal[target].sum()),\n             \"validation_pos\": int(val[target].sum()), \"validation_selectable\": val_all.height,\n             \"validation_unknown\": val_all.height - val.height}\n        fr: dict[str, Any] = {\"rows\": n, \"models\": {}}\n        res[\"folds\"][fold[\"name\"]] = fr\n        if min(n[\"train_pos\"], n[\"calibration_pos\"]) < 5 or n[\"validation_pos\"] < 1:\n            fr[\"skipped\"] = \"слишком мало положительных для обучения/калибровки\"\n            continue\n        tr = sample_train(pl, train, target, cfg)\n        y_cal, y_val = cal[target].to_numpy().astype(int), val[target].to_numpy().astype(int)\n        days = np.array([(d - date(2019, 1, 1)).days for d in val[\"d_cutoff_date\"].to_list()])\n        weeks = days // 7\n        days_all = np.array([(d - date(2019, 1, 1)).days for d in val_all[\"d_cutoff_date\"].to_list()])\n        chans_all = val_all[\"d_channel_key\"].to_numpy()\n        y_all = val_all[target].fill_null(-1).to_numpy().astype(int)\n        known_mask = y_all >= 0\n        prior = float(train[target].mean())\n        raws: dict[str, np.ndarray] = {}\n        fitted: dict[str, Any] = {}\n        no_cal = [c for c in features if c not in CALENDAR]\n        no_cat = [c for c in features if c not in CATALOGUE]\n        for name in MODELS:\n            t0 = time.time()\n            if name == \"P0_type_prior\":\n                m = TypePrior().fit(tr, target)\n                is_p = True\n            elif name == \"R1_rule\":\n                m, is_p = None, False\n            elif name == \"L1_logistic\":\n                m, is_p = Logistic().fit(tr, target, features, cfg), True\n            else:\n                feats = {\"G1_lightgbm\": features, \"G2_no_calendar\": no_cal, \"G3_no_catalogue\": no_cat}[name]\n                m, is_p = LightGBM().fit(tr, target, feats, cfg), True\n            score = (lambda f: rule_score(f, spec)) if m is None else m.score\n            raw_cal, raw_all = score(cal), score(val_all)\n            raw_val = raw_all[known_mask]\n            cal_model = platt(raw_cal, y_cal, is_p)\n            p_val = apply_platt(cal_model, raw_val, is_p, prior)\n            raws[name] = raw_val\n            fitted[name] = m\n            fr[\"models\"][name] = evaluate(y_val, raw_val, p_val, (days_all, chans_all, y_all, raw_all), cfg)\n            fr[\"models\"][name][\"fit_s\"] = round(time.time() - t0, 1)\n        fr[\"prevalence_validation\"] = float(y_val.mean())\n        fr[\"unknown_share_selectable\"] = n[\"validation_unknown\"] / max(n[\"validation_selectable\"], 1)\n        comp = spec.get(\"competing\")\n        if comp and comp in val.columns:\n            from sklearn.metrics import average_precision_score\n            # C5 / GPT 16: estimand «любой onset в D+2» против «первого» (onset в D+1 — отдельный исход), строки не удаляются\n            c = val[comp].fill_null(False).to_numpy().astype(bool)\n            y_first = (y_val == 1) & ~c\n            fr[\"competing_d1\"] = {\"share_of_rows\": float(c.mean()), \"share_of_positives\": float(c[y_val == 1].mean()) if y_val.sum() else None,\n                                  \"pr_auc_first_onset\": {k: (float(average_precision_score(y_first, raws[k]))\n                                                             if y_first.any() else None) for k in MODELS}}\n        types = val[\"тип_датчика\"].fill_null(\"нет в справочнике\").to_numpy()\n        fr[\"by_sensor_type_pr_auc\"] = {k: by_type_pr_auc(y_val, raws[k], types, cfg[\"min_type_positives\"]) for k in MODELS}\n        fr[\"importance_G1\"] = fitted[\"G1_lightgbm\"].importance()\n        best_base = max(BASELINES_FOR_GATE, key=lambda k: fr[\"models\"][k][\"pr_auc\"] or -1)\n        fr[\"best_simple_baseline\"] = best_base\n        for cand in (\"G1_lightgbm\", \"G2_no_calendar\", \"L1_logistic\"):\n            fr[f\"bootstrap_{cand}_vs_{best_base}\"] = paired_bootstrap(\n                y_val, raws[cand], raws[best_base], weeks, cfg[\"bootstrap_reps\"], cfg[\"random_seed\"])\n        log(f\"  {fold['name']}: val {n['validation']:,} строк, 1: {n['validation_pos']:,}; PR-AUC \"\n            + \", \".join(f\"{k.split('_')[0]} {fr['models'][k]['pr_auc']:.3f}\" for k in MODELS if fr['models'][k]['pr_auc'] is not None))\n    res[\"decision\"] = decide(res, cfg)\n    last = res[\"folds\"].get(FOLDS[-1][\"name\"], {})\n    if stress is not None and stress.height and \"skipped\" not in last and last.get(\"models\"):\n        from sklearn.metrics import average_precision_score\n        y_s = stress[target].to_numpy().astype(int)\n        out = {\"rows\": stress.height, \"positives\": int(y_s.sum()), \"prevalence\": float(y_s.mean()), \"pr_auc\": {}}\n        if 0 < y_s.sum() < len(y_s):\n            for name, m in fitted.items():\n                raw = rule_score(stress, spec) if m is None else m.score(stress)\n                out[\"pr_auc\"][name] = float(average_precision_score(y_s, raw))\n        res[\"stress_2021\"] = out\n    return res\n\n\ndef decide(res: dict[str, Any], cfg: dict[str, Any]) -> dict[str, Any]:\n    folds = list(res[\"folds\"].values())\n    gates: dict[str, Any] = {}\n    gates[\"g1_enough_validation_events\"] = all(f[\"rows\"][\"validation_pos\"] >= cfg[\"min_val_positives\"] and \"skipped\" not in f\n                                               for f in folds)\n    if not gates[\"g1_enough_validation_events\"]:\n        return {\"gates\": gates, \"selected\": None, \"evidence_level\": \"insufficient_events\",\n                \"abstain_reason\": \"too_few_validation_events\"}\n\n    def beats(model: str) -> bool:\n        return all((f[f\"bootstrap_{model}_vs_{f['best_simple_baseline']}\"][\"ci95_low\"] or -1) > 0 for f in folds)\n\n    def ratio(model: str) -> float:\n        return min(f[\"models\"][model][\"pr_auc\"] / max(f[\"models\"][\"G1_lightgbm\"][\"pr_auc\"], 1e-12) for f in folds)\n\n    tol = 1 - cfg[\"shortcut_tolerance\"]\n    gates[\"g2_G1_beats_simple_baselines_both_folds\"] = beats(\"G1_lightgbm\")\n    gates[\"g3_no_calendar_shortcut\"] = ratio(\"G2_no_calendar\") >= tol\n    gates[\"g4_no_catalogue_shortcut\"] = ratio(\"G3_no_catalogue\") >= tol\n    base_means = {k: float(np.mean([f[\"models\"][k][\"pr_auc\"] for f in folds])) for k in BASELINES_FOR_GATE}\n\n    def stable(model: str) -> bool | None:\n        \"\"\"≥ 2/3 типов датчиков с поддержкой: модель лучше лучшего простого baseline в каждом фолде.\n        Неприменимо (None), если в фолдах меньше двух типов с поддержкой (например, газовая цель T4).\"\"\"\n        applicable = False\n        for f in folds:\n            base = f[\"by_sensor_type_pr_auc\"][f[\"best_simple_baseline\"]]\n            mine = f[\"by_sensor_type_pr_auc\"][model]\n            common = [k for k in mine if k in base]\n            if len(common) < 2:\n                continue\n            applicable = True\n            if sum(mine[k] > base[k] for k in common) < 2 / 3 * len(common):\n                return False\n        return True if applicable else None\n\n    gates[\"g5_logistic_beats_simple_baselines_both_folds\"] = beats(\"L1_logistic\")\n    mean_pr = {k: float(np.mean([f[\"models\"][k][\"pr_auc\"] for f in folds])) for k in (\"G1_lightgbm\", \"G2_no_calendar\", \"L1_logistic\")}\n    if gates[\"g2_G1_beats_simple_baselines_both_folds\"]:\n        # если без календаря модель тоже побеждает baseline, а календарь даёт подозрительно много — берём G2\n        selected = \"G2_no_calendar\" if (not gates[\"g3_no_calendar_shortcut\"] and beats(\"G2_no_calendar\")) else \"G1_lightgbm\"\n        # при равенстве (±0,005 PR-AUC) предпочитаем более простую и устойчивую логистическую регрессию\n        if gates[\"g5_logistic_beats_simple_baselines_both_folds\"] and mean_pr[\"L1_logistic\"] >= mean_pr[selected] - 0.005:\n            selected = \"L1_logistic\"\n        level = \"validated_proxy_model\"\n    elif gates[\"g5_logistic_beats_simple_baselines_both_folds\"]:\n        selected, level = \"L1_logistic\", \"validated_proxy_model\"\n    else:\n        selected = max(base_means, key=base_means.get)\n        level = \"baseline_only\"\n    gates[\"g6_known_outcome_share\"] = all(1 - f[\"unknown_share_selectable\"] >= cfg[\"min_known_share_selectable\"] for f in folds)\n    gates[\"g7_stable_across_sensor_types\"] = stable(selected) if level == \"validated_proxy_model\" else None\n    return {\"gates\": gates, \"selected\": selected, \"evidence_level\": level, \"abstain_reason\": None,\n            \"simple_baseline_mean_pr_auc\": base_means}\n\n\ndef score_kind(spec: dict[str, Any], decision: dict[str, Any], folds: dict[str, Any], cfg: dict[str, Any]) -> str:\n    \"\"\"Что отдаёт сервис: калиброванную вероятность обещаем только при подтверждённой модели и ECE ≤ порога.\"\"\"\n    if decision[\"evidence_level\"] == \"insufficient_events\":\n        return \"abstain\"\n    if spec[\"family\"] == \"conditional_proxy_score\":\n        return \"conditional_proxy_score\"  # цензура длительности: на всю популяцию не калибруем (согласовано 10/Q1)\n    sel = decision[\"selected\"]\n    eces = [f[\"models\"][sel][\"ece10\"] for f in folds.values() if sel in f.get(\"models\", {})]\n    unknown = [f.get(\"unknown_share_selectable\", 1.0) for f in folds.values()]\n    # ECE на наблюдаемых строках не доказывает калибровку для всех каналов: требуем ещё малую долю unknown (GPT 14)\n    if (decision[\"evidence_level\"] == \"validated_proxy_model\" and eces and max(eces) <= cfg[\"ece_calibrated_max\"]\n            and max(unknown) <= cfg[\"max_unknown_share_calibrated\"]):\n        return \"probability_calibrated\"\n    return \"proxy_score\"\n\n\ndef backend_contract(results: dict[str, Any], cov: dict[str, Any], cfg: dict[str, Any]) -> dict[str, Any]:\n    out = {\"schema_version\": SCHEMA_VERSION, \"mode\": cfg[\"mode\"],\n           \"window\": \"метка: события в [D+2; D+3) по признакам на конец D; решение метки не позже D+4\",\n           \"cooldown_days\": cfg[\"cooldown_days\"], \"targets\": [],\n           \"data_quality_flags\": [{\"tz_item\": \"T1b\", \"incident_type\": \"clock_error_1970\", \"score_kind\": \"data_quality_flag\",\n                                   \"source\": \"значение 01.01.1970 в журнале (FAQ: «сбилась дата»); признаки f_n_clock_1970_*\",\n                                   \"not_a_failure_label\": True}],\n           \"evidence_levels\": {\"validated_proxy_model\": \"модель лучше простых baseline в обоих фолдах (95% CI)\",\n                               \"baseline_only\": \"отдаётся скор простого правила/приора\",\n                               \"insufficient_events\": \"мало событий — abstain\"}}\n    for t, r in results.items():\n        spec = TARGET_SPECS[t]\n        d = r[\"decision\"]\n        last = r[\"folds\"].get(\"fold_2024\", {})\n        sel = d.get(\"selected\")\n        m = last.get(\"models\", {}).get(sel, {}) if sel else {}\n        reasons = cov[\"targets\"][t][\"reasons\"]\n        abstain = {k: v for k, v in sorted(reasons.items(), key=lambda kv: -kv[1])\n                   if k not in (\"positive\", \"negative\", \"negative_transient_only\")}\n        entry = {\n            \"target\": t, \"tz_item\": spec[\"tz\"], \"incident_type\": spec[\"incident_type\"],\n            \"score_kind\": score_kind(spec, d, r[\"folds\"], cfg),\n            \"evidence_level\": d[\"evidence_level\"], \"selected_model\": sel,\n            \"coverage_labelled_share\": round(cov[\"targets\"][t][\"labelled_share\"], 4),\n            \"abstain_reasons_rows\": abstain,\n            \"validation_fold_2024\": ({\"pr_auc_on_known\": m.get(\"pr_auc\"), \"prevalence_on_known\": last.get(\"prevalence_validation\"),\n                                      \"unknown_share_selectable\": last.get(\"unknown_share_selectable\"),\n                                      \"brier\": m.get(\"brier\"), \"ece10\": m.get(\"ece10\"),\n                                      \"budget_25_per_day\": m.get(\"budget\", {}).get(\"25\")} if m else None),\n            \"not_a_confirmed_failure\": True,\n        }\n        if spec[\"family\"] == \"conditional_proxy_score\":\n            lab = cov[\"targets\"][t][\"labelled_rows\"]\n            cen = reasons.get(\"censored_duration\", 0)\n            entry[\"censoring\"] = {\"censored_rows\": cen, \"censored_share_of_decidable\": cen / max(lab + cen, 1),\n                                  \"note\": \"метрики — по нецензурированным строкам; для всей популяции это bounds, не оценка\"}\n        out[\"targets\"].append(entry)\n    return out\n\n\ndef run_targets_v3(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    import polars as pl\n\n    t_start = time.time()\n    out = Path(cfg[\"output_dir\"])\n    out.mkdir(parents=True, exist_ok=True)\n    parts, manifest = discover_panel(Path(cfg[\"input_dir\"]), cfg[\"verify_sha256\"])\n    columns = pl.scan_parquet(parts).collect_schema().names()\n    features = safe_features(manifest, columns)\n    log(f\"панель: {manifest['rows']:,} строк, частей {len(parts)}, признаков {len(features)}; режим {cfg['mode']}\")\n    cov = coverage(pl, parts, cfg[\"targets\"], cfg)\n    results = {}\n    for t in cfg[\"targets\"]:\n        log(f\"{t}:\")\n        frame = load_target(pl, parts, t, features, cfg)\n        stress = load_stress(pl, parts, t, features, cfg)\n        results[t] = run_target(pl, frame, t, features, cfg, log, stress=stress)\n        del stress\n        del frame\n    contract = backend_contract(results, cov, cfg)\n    status = \"completed_smoke_non_comparable\" if cfg[\"mode\"] == \"SMOKE\" else \"completed\"\n    payload = {\"schema_version\": SCHEMA_VERSION, \"status\": status, \"mode\": cfg[\"mode\"],\n               \"panel_manifest\": {\"rows\": manifest[\"rows\"], \"taxonomy_sha256\": manifest[\"taxonomy_sha256\"],\n                                  \"parts_sha256\": [p[\"sha256\"] for p in manifest[\"parts\"]],\n                                  \"panel_config\": manifest[\"panel_config\"]},\n               \"protocol\": {\"folds\": FOLDS,\n                            \"excluded_cutoff_dates\": [str(x) for x in EXCLUDED], \"load_end_exclusive\": str(LOAD_END),\n                            \"features\": features, \"models\": MODELS},\n               \"config\": {k: v for k, v in cfg.items() if k not in (\"input_dir\", \"output_dir\")},\n               \"coverage\": cov, \"targets\": results, \"runtime_s\": round(time.time() - t_start, 1)}\n    (out / \"results_targets_v3.json\").write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding=\"utf-8\")\n    (out / \"backend_typed_scores_v3.json\").write_text(json.dumps(contract, ensure_ascii=False, indent=2, default=str),\n                                                      encoding=\"utf-8\")\n    (out / \"summary_targets_v3_ru.md\").write_text(summary_markdown(payload, contract), encoding=\"utf-8\")\n    return {\"status\": status, \"runtime_s\": payload[\"runtime_s\"],\n            \"decisions\": {t: r[\"decision\"][\"evidence_level\"] for t, r in results.items()}}\n\n\ndef _f(x: Any, d: int = 3) -> str:\n    return \"—\" if x is None else f\"{x:.{d}f}\"\n\n\ndef summary_markdown(payload: dict[str, Any], contract: dict[str, Any]) -> str:\n    L = [f\"# Ноутбук 15 — модели на целях панели v3 ({payload['mode']})\", \"\"]\n    if payload[\"mode\"] == \"SMOKE\":\n        L += [\"**SMOKE:** часть каналов, мало деревьев — цифры не сравнимы с FULL.\", \"\"]\n    L += [\"Метки — наблюдаемые события журнала, не подтверждённые отказы. 2021 и январь 2022 исключены; \"\n          \"2025+ не загружается.\", \"\", \"## Итог по целям\", \"\",\n          \"| цель | ТЗ | evidence_level | score_kind | модель | покрытие строк | PR-AUC 2023 | PR-AUC 2024 | unknown 2024 | precision@25/сут (cooldown) 2024, границы |\",\n          \"|---|---|---|---|---|---|---|---|---|---|\"]\n    for c in contract[\"targets\"]:\n        r = payload[\"targets\"][c[\"target\"]]\n        sel = c[\"selected_model\"]\n        pr = [_f(r[\"folds\"][f].get(\"models\", {}).get(sel, {}).get(\"pr_auc\")) if sel else \"—\" for f in (\"fold_2023\", \"fold_2024\")]\n        v = c[\"validation_fold_2024\"] or {}\n        b = v.get(\"budget_25_per_day\") or {}\n        L.append(f\"| `{c['target']}` | {c['tz_item']} | **{c['evidence_level']}** | {c['score_kind']} | {sel or '—'} | \"\n                 f\"{c['coverage_labelled_share']:.3f} | {pr[0]} | {pr[1]} | {_f(v.get('unknown_share_selectable'))} | \"\n                 f\"{_f(b.get('precision_cooldown'))}–{_f(b.get('precision_upper_cooldown'))} |\")\n    for t, r in payload[\"targets\"].items():\n        L += [\"\", f\"## `{t}`\", \"\"]\n        for fname, f in r[\"folds\"].items():\n            L += [f\"**{fname}**: train {f['rows']['train']:,} (1: {f['rows']['train_pos']:,}), \"\n                  f\"validation {f['rows']['validation']:,} (1: {f['rows']['validation_pos']:,})\", \"\"]\n            if \"skipped\" in f:\n                L += [f\"пропущен: {f['skipped']}\", \"\"]\n                continue\n            L += [f\"selectable в валидации: {f['rows']['validation_selectable']:,}, исход неизвестен: \"\n                  f\"{f['rows']['validation_unknown']:,} ({f['unknown_share_selectable']:.1%}). PR-AUC/ROC/Brier/ECE — по известным; \"\n                  \"P — нижняя–верхняя граница по всей selectable когорте, R — по известным положительным.\", \"\",\n                  \"| модель | PR-AUC | ROC-AUC | Brier | ECE | P@10 | R@10 | P@25 cd | R@25 cd | слотов в unknown @25 cd |\",\n                  \"|---|---|---|---|---|---|---|---|---|---|\"]\n            for m, x in f[\"models\"].items():\n                b10, b25 = x[\"budget\"].get(\"10\", {}), x[\"budget\"].get(\"25\", {})\n                L.append(f\"| {m} | {_f(x['pr_auc'])} | {_f(x['roc_auc'])} | {_f(x['brier'], 4)} | {_f(x['ece10'], 4)} | \"\n                         f\"{_f(b10.get('precision'))}–{_f(b10.get('precision_upper'))} | {_f(b10.get('recall'))} | \"\n                         f\"{_f(b25.get('precision_cooldown'))}–{_f(b25.get('precision_upper_cooldown'))} | \"\n                         f\"{_f(b25.get('recall_cooldown'))} | {b25.get('unknown_alerts_cooldown', 0):,} |\")\n            bb = f[\"best_simple_baseline\"]\n            bs = f[f\"bootstrap_G1_lightgbm_vs_{bb}\"]\n            bl = f[f\"bootstrap_L1_logistic_vs_{bb}\"]\n            L += [\"\", f\"G1 − {bb}: ΔPR-AUC {_f(bs['mean'])} [{_f(bs['ci95_low'])}; {_f(bs['ci95_high'])}], повторов {bs['reps']}; \"\n                  f\"L1 − {bb}: {_f(bl['mean'])} [{_f(bl['ci95_low'])}; {_f(bl['ci95_high'])}].\",\n                  \"Топ признаков G1: \" + \", \".join(f\"{k} {v:.2f}\" for k, v in list(f[\"importance_G1\"].items())[:6]), \"\"]\n            b50 = f[\"models\"][\"G1_lightgbm\"][\"budget\"].get(\"50\", {})\n            L += [f\"Top-50/сутки, G1, cooldown: слотов в unknown {b50.get('unknown_alerts_cooldown', 0):,} из {b50.get('alerts_cooldown', 0):,}; \"\n                  f\"recall [{_f(b50.get('recall_low_cooldown'))}; {_f(b50.get('recall_up_cooldown'))}], \"\n                  f\"precision [{_f(b50.get('precision_cooldown'))}; {_f(b50.get('precision_upper_cooldown'))}].\", \"\"]\n            if \"competing_d1\" in f:\n                cd = f[\"competing_d1\"]\n                L += [f\"Onset в D+1 (competing): {_f(cd['share_of_rows'])} строк, {_f(cd['share_of_positives'])} положительных. \"\n                      \"PR-AUC для «первого onset» (onset в D+2 без onset в D+1): \"\n                      + \", \".join(f\"{k.split('_')[0]} {_f(v)}\" for k, v in cd[\"pr_auc_first_onset\"].items()), \"\"]\n        d = r[\"decision\"]\n        if r.get(\"stress_2021\", {}).get(\"pr_auc\"):\n            st = r[\"stress_2021\"]\n            L += [f\"Stress 2021 (модели fold_2024, в выборе не участвует): строк {st['rows']:,}, prevalence {_f(st['prevalence'], 4)}; PR-AUC \"\n                  + \", \".join(f\"{k.split('_')[0]} {_f(v)}\" for k, v in st[\"pr_auc\"].items()), \"\"]\n        L += [\"Gates: \" + \", \".join(f\"{k} = {'—' if v is None else ('да' if v else 'нет')}\" for k, v in d[\"gates\"].items()), \"\"]\n    L += [\"\", \"## Как читать\", \"\",\n          \"- `validated_proxy_model` — модель (LightGBM или логистическая регрессия) лучше простых baseline (приор по типу, правило) \"\n          \"в обоих фолдах: нижняя граница 95% CI разницы PR-AUC > 0; при равенстве берётся логистическая регрессия.\",\n          \"- `baseline_only` — модель не доказала преимущества; бэкенд отдаёт скор простого правила/приора.\",\n          \"- `insufficient_events` — мало событий в валидации: бэкенд не выдаёт скор (abstain).\",\n          \"- P@k — precision топ-k строк в сутки среди размеченных строк; `cd` — с cooldown по каналу.\",\n          \"\"]\n    return \"\\n\".join(L)\n\n\n# ---------------------------------------------------------------------------\n# Синтетика для самопроверок и тестов\n# ---------------------------------------------------------------------------\ndef synthetic_panel_v3(out_dir: Path, n_channels: int = 150, seed: int = 3, signal: float = 2.0,\n                       end: date = date(2025, 1, 10), gas_features: bool = False) -> Path:\n    \"\"\"Панель со схемой v3 и встроенным сигналом (для тестов; не реальные данные).\n    gas_features: добавить f_gas_n_7d / f_gas_last и строки above_threshold_at_D для T4 (тесты экспорта 16).\"\"\"\n    import polars as pl\n\n    rng = np.random.default_rng(seed)\n    days = [date(2019, 1, 1) + timedelta(days=i) for i in range(0, (end - date(2019, 1, 1)).days, 3)]\n    n = n_channels * len(days)\n    ch = np.repeat(np.arange(n_channels), len(days))\n    d = np.tile(np.array(days, dtype=\"datetime64[D]\"), n_channels)\n    types = np.array([\"Состояние насоса\", \"КД Дверь\", \"Газовый датчик\", \"Состояние фазы\"])[ch % 4]\n    base = rng.poisson(0.3, n).astype(float)\n    frame = {\n        \"d_channel_key\": np.array([hashlib.sha256(f\"s{c}\".encode()).hexdigest()[:16] for c in range(n_channels)])[ch],\n        \"d_cutoff_date\": d, \"d_label_decision_end\": d + np.timedelta64(4, \"D\"),\n        \"тип_датчика\": types, \"тип_инж_системы\": np.where(ch % 2 == 0, \"ВК\", \"ЭС\"),\n        \"link_state\": rng.choice([\"O\", \"F\", \"A\"], n, p=[0.9, 0.05, 0.05]),\n        \"power_state\": rng.choice([\"O\", \"F\"], n, p=[0.9, 0.1]),\n        \"d_weekday\": (d.astype(\"datetime64[D]\").astype(int) + 3) % 7 + 1,\n        \"d_month\": d.astype(\"datetime64[M]\").astype(int) % 12 + 1,\n        \"f_n_events_7d\": rng.poisson(50, n).astype(float),\n    }\n    for t, spec in TARGET_SPECS.items():\n        for col in spec[\"rule\"]:\n            frame.setdefault(col, base + rng.poisson(0.2, n))\n    noise = rng.normal(0, 1, n)\n    for t, spec in TARGET_SPECS.items():\n        x = frame[spec[\"rule\"][0]]\n        logit = -4.0 + signal * (x - x.mean()) / (x.std() + 1e-9) + 0.3 * noise\n        y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(float)\n        null = rng.random(n) < 0.3\n        frame[t] = np.where(null, np.nan, y)\n        frame[\"reason_\" + t[len(\"target_\"):]] = np.where(null, \"unobserved_window\", np.where(y == 1, \"positive\", \"negative\"))\n    if gas_features:\n        # объект = 8 каналов; ежегодное «молчание» объекта (как ППР): gas_max = null 12 суток в году\n        obj = ch // 8\n        frame[\"d_object_key\"] = np.array([hashlib.sha256(f\"o{o}\".encode()).hexdigest()[:16] for o in range(n_channels // 8 + 1)])[obj]\n        doy = (d.astype(\"datetime64[D]\") - d.astype(\"datetime64[Y]\")).astype(int)\n        silent = np.abs(doy - (40 + 23 * (obj % 12))) < 6\n        frame[\"gas_max\"] = np.where(silent, np.nan, rng.uniform(0, 0.9, n))\n        frame[\"f_gas_cross_n_30d\"] = rng.poisson(0.3, n).astype(float)\n        above = rng.random(n) < 0.1\n        frame[\"f_gas_n_7d\"] = rng.poisson(5, n).astype(float) + 1\n        frame[\"f_gas_last\"] = np.where(above, 1.5, rng.uniform(0, 0.9, n))\n        frame[\"target_t4_gas_cross\"] = np.where(above, np.nan, frame[\"target_t4_gas_cross\"])\n        frame[\"reason_t4_gas_cross\"] = np.where(above, \"above_threshold_at_D\", frame[\"reason_t4_gas_cross\"])\n    panel = pl.DataFrame(frame).with_columns(\n        pl.col(\"d_cutoff_date\").cast(pl.Date), pl.col(\"d_label_decision_end\").cast(pl.Date),\n        *[pl.col(t).fill_nan(None).cast(pl.Int8) for t in TARGET_SPECS])\n    panel_dir = Path(out_dir) / \"event_panel_v3\"\n    panel_dir.mkdir(parents=True, exist_ok=True)\n    part = panel_dir / \"event_panel_v3_part00.parquet\"\n    panel.write_parquet(part)\n    feats = [c for c in panel.columns if c.startswith(\"f_\")] + CATEGORICAL + CALENDAR\n    manifest = {\"schema_version\": SCHEMA_VERSION, \"rows\": panel.height, \"taxonomy_sha256\": \"synthetic\",\n                \"panel_config\": {\"synthetic\": True}, \"feature_columns\": feats, \"target_columns\": list(TARGET_SPECS),\n                \"parts\": [{\"file\": part.name, \"sha256\": file_sha256(part), \"rows\": panel.height}]}\n    (panel_dir / \"panel_manifest_v3.json\").write_text(json.dumps(manifest), encoding=\"utf-8\")\n    return panel_dir\n\n\ndef run_targets_self_tests(work_dir: Path) -> dict[str, bool]:\n    import polars as pl\n\n    work_dir = Path(work_dir)\n    rep: dict[str, bool] = {}\n    y = np.array([0, 1, 0, 1, 0, 0])\n    days = np.array([0, 0, 0, 1, 1, 1])\n    ch = np.array([\"a\", \"b\", \"c\", \"b\", \"a\", \"c\"])\n    s = np.array([0.1, 0.9, 0.2, 0.8, 0.3, 0.1])\n    b = budget_curve(days, ch, y, s, [1], cooldown_days=3)[\"1\"]\n    rep[\"budget_top1_hits_both_days\"] = b[\"precision\"] == 1.0 and b[\"recall\"] == 1.0\n    rep[\"cooldown_blocks_repeat_channel\"] = b[\"alerts_cooldown\"] == 2 and b[\"recall_cooldown\"] == 0.5\n    bu = budget_curve(np.array([0, 0]), np.array([\"a\", \"b\"]), np.array([-1, 1]), np.array([0.9, 0.1]), [1], 3)[\"1\"]\n    rep[\"unknown_slot_counted_as_bounds\"] = (bu[\"precision\"] == 0.0 and bu[\"precision_upper\"] == 1.0 and bu[\"unknown_alerts\"] == 1\n                                             and bu[\"recall_low\"] == 0.0 and bu[\"recall_up\"] == 0.5)\n    try:\n        safe_features({\"feature_columns\": [\"f_x\", \"competing_link_onset_d1\"]}, [\"f_x\", \"competing_link_onset_d1\"])\n        rep[\"future_columns_rejected\"] = False\n    except ValueError:\n        rep[\"future_columns_rejected\"] = True\n    frame = pl.DataFrame({\"d_cutoff_date\": [date(2022, 12, 25), date(2022, 12, 30), date(2023, 6, 29), date(2023, 7, 3)],\n                          \"d_label_decision_end\": [date(2022, 12, 29), date(2023, 1, 3), date(2023, 7, 3), date(2023, 7, 7)]})\n    tr, ca, va = split_fold(pl, frame, FOLDS[0])\n    rep[\"purging_by_label_decision_end\"] = tr.height == 1 and ca.height == 0 and va.height == 1\n    root = work_dir / \"synthetic_in\"\n    synthetic_panel_v3(root, n_channels=60)\n    cfg = make_config_targets(\"SMOKE\", {\"input_dir\": str(root), \"output_dir\": str(work_dir / \"out\"),\n                                        \"smoke_channel_share\": 1, \"targets\": [\"target_t2a_link_onset\"],\n                                        \"bootstrap_reps\": 10})\n    res = run_targets_v3(cfg, log=lambda *a: None)\n    rep[\"synthetic_end_to_end\"] = res[\"status\"] == \"completed_smoke_non_comparable\"\n    text = (work_dir / \"out\" / \"results_targets_v3.json\").read_text(encoding=\"utf-8\")\n    keys = pl.read_parquet(root / \"event_panel_v3\" / \"event_panel_v3_part00.parquet\")[\"d_channel_key\"].unique().to_list()\n    rep[\"no_channel_keys_in_outputs\"] = not any(k in text for k in keys)\n    return rep\n",
"maintenance_audit_v3.py": "\"\"\"Ноутбук 17: read-only audit переданного газового bundle внутри/вне окон сервисных работ (без обучения).\n\nСпецификация согласована заранее (Drive 38–44, docs/AUDIT_17_MAINTENANCE_RU.md):\n- скорится НЕИЗМЕНЁННЫЙ gas_cross_v3_bundle (predictor.py + model.txt) и правило R1 на одних и тех же строках;\n- периоды: 2025 H2 и 2026 H1 — оба уже исследовались проектом, это ретроспективный audit, не нетронутый тест;\n- окна работ двух видов, строго раздельно:\n  (a) post-hoc: у объекта ВСЕ газовые каналы без показаний ≥ min_silence суток подряд → окно\n      [начало молчания − before; конец молчания + after]; использует будущее, только для исследования меток;\n  (b) причинный trailing-флаг на D: такое молчание закончилось в [D − lookback; D − 1] и возврат уже наблюдался к D;\n- строка попадает в окно, если её день исхода D+2 лежит в окне; цензура симметричная (y = 1, 0 или unknown);\n- чувствительность after = 14 / 21 / 30; negative control — те же окна, сдвинутые на +30 и +60 суток;\n- метрики на каждой страте: counts, unknown, prevalence, PR-AUC и lift (PR-AUC / prevalence) для G2 и R1, Brier/ECE G2,\n  блочный бутстрэп по неделям ΔPR-AUC (G2 − R1); top-25/сутки — ОДНА общая политика с cooldown, разбитая по стратам;\n- никаких порогов, выбора модели или изменения bundle; в выходах только агрегаты.\n\nv2 (после проверки GPT, Drive 46):\n- preflight: ZIP обязан совпасть с .sha256, а manifest панели, число строк, схема и SHA частей — с тем, что записано\n  в bundle.json; иначе PanelMismatch (скорить старой моделью другую панель нельзя);\n- молчание считается только между двумя реальными газовыми показаниями объекта внутри наблюдаемого архива\n  (нет искусственных интервалов до первого появления объекта и после конца архива);\n- окна присваиваются строкам через стабильный индекс строки;\n- top-25 и cooldown для G2 и R1 — на одной и той же скорируемой eligible-когорте;\n- контрольные сдвинутые окна не пересекаются с основным post-hoc окном.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport shutil\nimport sys\nimport time\nimport zipfile\nfrom datetime import date, timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nimport target_models_v3_runtime as rt\n\nTARGET = \"target_t4_gas_cross\"\nGAS_TYPE = \"Газовый датчик\"\nPERIODS = {\n    \"2025H2\": (date(2025, 7, 1), date(2026, 1, 1)),\n    \"2026H1\": (date(2026, 1, 1), date(2026, 7, 1)),\n}\nCONTROL_SHIFTS = (30, 60)\nAFTER_VARIANTS = (14, 21, 30)\n\n\ndef make_config_17(mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:\n    cfg = rt.make_config_targets(mode)\n    cfg.update({\"periods\": list(PERIODS), \"min_silence_days\": 4, \"before_days\": 3, \"after_days\": 21,\n                \"lookback_days\": 21, \"budget_per_day\": 25, \"bootstrap_reps\": 200 if mode == \"FULL\" else 30,\n                \"min_positives_for_bootstrap\": 30, \"clean_history_col\": \"f_gas_cross_n_30d\", \"verify_sha256\": True})\n    cfg.update(overrides or {})\n    return cfg\n\n\n# ---------------------------------------------------------------------------\n# Bundle\n# ---------------------------------------------------------------------------\ndef _sha256(path: Path) -> str:\n    return rt.file_sha256(Path(path))\n\n\nclass PanelMismatch(ValueError):\n    \"\"\"Подключённая панель — не та, на которой обучен bundle.\"\"\"\n\n\ndef check_panel_matches_bundle(manifest: dict[str, Any], manifest_path: Path, spec: dict[str, Any]) -> dict[str, Any]:\n    \"\"\"Строгая сверка панели с bundle.json: SHA manifest, схема, число строк, SHA всех частей.\"\"\"\n    exp = spec.get(\"panel\") or {}\n    got = {\"manifest_sha256\": _sha256(manifest_path), \"schema_version\": manifest.get(\"schema_version\"),\n           \"rows\": manifest.get(\"rows\"), \"parts_sha256\": [p[\"sha256\"] for p in manifest.get(\"parts\", [])]}\n    bad = [k for k in got if exp.get(k) != got[k]]\n    if bad:\n        raise PanelMismatch(\n            \"панель не совпадает с bundle по полям \" + \", \".join(bad)\n            + f\". Найдена панель: manifest_sha256={got['manifest_sha256']}, строк {got['rows']}, \"\n            + f\"путь входа {manifest_path.parent}. Ожидалась: manifest_sha256={exp.get('manifest_sha256')}, \"\n            + f\"строк {exp.get('rows')}. Подключите ту версию выхода 14, на которой обучался 16; проверку не обходить.\")\n    return {\"panel_matches_bundle\": True, \"manifest_sha256\": got[\"manifest_sha256\"], \"rows\": got[\"rows\"],\n            \"parts\": len(got[\"parts_sha256\"]), \"data_end\": manifest.get(\"data_end\")}\n\n\ndef find_bundle(input_dir: Path, work_dir: Path) -> tuple[Path, dict[str, Any]]:\n    \"\"\"Ищет gas_cross_v3_bundle.zip во входах; ZIP обязан совпасть с gas_cross_v3_bundle.zip.sha256 рядом.\"\"\"\n    input_dir = Path(input_dir)\n    info: dict[str, Any] = {}\n    zips = sorted(input_dir.rglob(\"gas_cross_v3_bundle.zip\"))\n    if zips:\n        z = zips[0]\n        info[\"zip_sha256\"] = _sha256(z)\n        side = list(z.parent.glob(\"gas_cross_v3_bundle.zip.sha256\"))\n        if not side:\n            raise FileNotFoundError(\"рядом с gas_cross_v3_bundle.zip нет gas_cross_v3_bundle.zip.sha256\")\n        expected = side[0].read_text(encoding=\"utf-8\").split()[0]\n        if expected != info[\"zip_sha256\"]:\n            raise ValueError(\"SHA-256 ZIP не совпадает с .sha256 — это не переданный bundle\")\n        info[\"zip_sha256_verified\"] = True\n        out = Path(work_dir) / \"_bundle17\"\n        if out.exists():\n            shutil.rmtree(out)\n        with zipfile.ZipFile(z) as f:\n            f.extractall(out)\n        found = sorted(out.rglob(\"bundle.json\"))\n    else:\n        raise FileNotFoundError(\"gas_cross_v3_bundle.zip не найден: подключите dataset с ZIP и .zip.sha256\")\n    if not found:\n        raise FileNotFoundError(\"в gas_cross_v3_bundle.zip нет bundle.json\")\n    return found[0].parent, info\n\n\ndef load_predictor(bundle_dir: Path) -> Any:\n    spec = importlib.util.spec_from_file_location(\"gas_bundle_predictor_17\", Path(bundle_dir) / \"predictor.py\")\n    module = importlib.util.module_from_spec(spec)\n    flag, sys.dont_write_bytecode = sys.dont_write_bytecode, True\n    try:\n        spec.loader.exec_module(module)\n    finally:\n        sys.dont_write_bytecode = flag\n    return module\n\n\n# ---------------------------------------------------------------------------\n# Данные\n# ---------------------------------------------------------------------------\ndef load_period(pl: Any, parts: list[Path], names: list[str], cfg: dict[str, Any], period: tuple[date, date]) -> Any:\n    \"\"\"Selectable строки T4 периода (метка известна или unknown-причина), как в 15/16.\"\"\"\n    reason = \"reason_\" + TARGET[len(\"target_\"):]\n    extra = [c for c in dict.fromkeys(list(rt.TARGET_SPECS[TARGET][\"rule\"]) + [cfg[\"clean_history_col\"]]) if c not in names]\n    lf = pl.scan_parquet(parts)\n    cols = lf.collect_schema().names()\n    extra = [c for c in extra if c in cols]\n    lf = lf.filter((pl.col(\"d_cutoff_date\") >= period[0]) & (pl.col(\"d_cutoff_date\") < period[1])\n                   & (pl.col(\"d_label_decision_end\") <= period[1])\n                   & (pl.col(TARGET).is_not_null() | pl.col(reason).is_in(rt.UNKNOWN_REASONS)))\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    numeric = [c for c in names + extra if c not in rt.CATEGORICAL]\n    keep = [\"d_channel_key\", \"d_object_key\", \"d_cutoff_date\", TARGET, reason] + names + extra\n    return lf.select(keep).with_columns([pl.col(c).cast(pl.Float32) for c in numeric]).collect()\n\n\ndef object_gas_activity(pl: Any, parts: list[Path], start: date, end: date) -> Any:\n    \"\"\"(объект, сутки) → было ли хоть одно газовое показание (gas_max у газового канала). Полный календарь объекта:\n    сутки без строк панели считаются молчанием.\"\"\"\n    lf = (pl.scan_parquet(parts)\n          .filter((pl.col(\"тип_датчика\") == GAS_TYPE) & pl.col(\"d_object_key\").is_not_null()\n                  & (pl.col(\"d_cutoff_date\") >= start) & (pl.col(\"d_cutoff_date\") < end))\n          .select(\"d_object_key\", pl.col(\"d_cutoff_date\").alias(\"day\"),\n                  pl.col(\"gas_max\").cast(pl.Float64).fill_nan(None).is_not_null().alias(\"reading\")))\n    obs = lf.group_by(\"d_object_key\", \"day\").agg(pl.col(\"reading\").any()).collect()\n    days = pl.DataFrame({\"day\": pl.date_range(start, end - timedelta(days=1), eager=True)})\n    objs = obs.select(\"d_object_key\").unique()\n    return (objs.join(days, how=\"cross\").join(obs, on=[\"d_object_key\", \"day\"], how=\"left\")\n            .with_columns(pl.col(\"reading\").fill_null(False)).sort(\"d_object_key\", \"day\"))\n\n\ndef silence_runs(pl: Any, activity: Any, min_silence: int) -> Any:\n    \"\"\"Отрезки молчания всех газовых каналов объекта длиной ≥ min_silence суток: (объект, start, end, returned).\n\n    Учитываются только отрезки, ограниченные реальными показаниями С ОБЕИХ сторон: сутки до первого и после\n    последнего показания объекта в наблюдаемом окне не считаются молчанием (объект ещё не появился / архив кончился).\n    Поэтому returned всегда True; колонка оставлена для совместимости.\"\"\"\n    bounds = (activity.filter(pl.col(\"reading\")).group_by(\"d_object_key\")\n              .agg(pl.col(\"day\").min().alias(\"_first\"), pl.col(\"day\").max().alias(\"_lastread\")))\n    activity = (activity.join(bounds, on=\"d_object_key\", how=\"inner\")\n                .filter((pl.col(\"day\") >= pl.col(\"_first\")) & (pl.col(\"day\") <= pl.col(\"_lastread\")))\n                .drop(\"_first\", \"_lastread\").sort(\"d_object_key\", \"day\"))\n    # два шага: вложенное .over внутри .over не поддерживается в polars 1.31 (Kaggle)\n    a = (activity.with_columns((pl.col(\"reading\") != pl.col(\"reading\").shift(1).over(\"d_object_key\"))\n                               .fill_null(True).cast(pl.UInt32).alias(\"_chg\"))\n         .with_columns(pl.col(\"_chg\").cum_sum().over(\"d_object_key\").alias(\"_run\")))\n    runs = (a.filter(~pl.col(\"reading\")).group_by(\"d_object_key\", \"_run\")\n            .agg(pl.col(\"day\").min().alias(\"start\"), pl.col(\"day\").max().alias(\"end\"), pl.len().alias(\"length\")))\n    last_day = activity.group_by(\"d_object_key\").agg(pl.col(\"day\").max().alias(\"_last\"))\n    return (runs.filter(pl.col(\"length\") >= min_silence).join(last_day, on=\"d_object_key\")\n            .with_columns((pl.col(\"end\") < pl.col(\"_last\")).alias(\"returned\")).drop(\"_run\", \"_last\"))\n\n\ndef posthoc_days(pl: Any, runs: Any, before: int, after: int, shift: int = 0) -> Any:\n    \"\"\"Дни post-hoc окон (объект, день); shift > 0 — negative control того же размера.\"\"\"\n    w = runs.select(\"d_object_key\",\n                    (pl.col(\"start\") - pl.duration(days=before - shift)).alias(\"a\"),\n                    (pl.col(\"end\") + pl.duration(days=after + shift)).alias(\"b\"))\n    return (w.with_columns(pl.date_ranges(\"a\", \"b\").alias(\"day\")).explode(\"day\")\n            .select(\"d_object_key\", \"day\").unique())\n\n\ndef trailing_days(pl: Any, runs: Any, lookback: int) -> Any:\n    \"\"\"Причинный флаг: дни D, для которых молчание закончилось в [D − lookback; D − 1] и возврат наблюдался к D.\"\"\"\n    w = runs.filter(pl.col(\"returned\")).select(\n        \"d_object_key\", (pl.col(\"end\") + pl.duration(days=1)).alias(\"a\"), (pl.col(\"end\") + pl.duration(days=lookback)).alias(\"b\"))\n    return (w.with_columns(pl.date_ranges(\"a\", \"b\").alias(\"day\")).explode(\"day\")\n            .select(\"d_object_key\", \"day\").unique())\n\n\ndef mark(pl: Any, frame: Any, days: Any, name: str, offset_days: int) -> Any:\n    \"\"\"Флаг строки: (объект, D + offset_days) входит в набор дней. Выравнивание — по стабильному индексу строки,\n    а не по порядку строк после join.\"\"\"\n    key = frame.select(pl.int_range(0, pl.len(), dtype=pl.UInt32).alias(\"_row\"), \"d_object_key\",\n                       (pl.col(\"d_cutoff_date\") + pl.duration(days=offset_days)).alias(\"day\"))\n    hits = (key.join(days.select(\"d_object_key\", \"day\").unique(), on=[\"d_object_key\", \"day\"], how=\"semi\")\n            .get_column(\"_row\").to_numpy())\n    flag = np.zeros(frame.height, dtype=bool)\n    flag[hits] = True\n    return frame.with_columns(pl.Series(name, flag))\n\n\n# ---------------------------------------------------------------------------\n# Метрики\n# ---------------------------------------------------------------------------\ndef select_alerts(days: np.ndarray, chans: np.ndarray, score: np.ndarray, k: int, cooldown: int) -> np.ndarray:\n    \"\"\"Общая политика: top-k строк в сутки с cooldown по каналу; возвращает маску выданных тревог.\"\"\"\n    order = np.lexsort((-score, days))\n    picked = np.zeros(len(days), dtype=bool)\n    last: dict[Any, int] = {}\n    cur, taken = None, 0\n    for i in order:\n        d = int(days[i])\n        if d != cur:\n            cur, taken = d, 0\n        if taken >= k:\n            continue\n        ch = chans[i]\n        if ch in last and d - last[ch] < cooldown:\n            continue\n        last[ch] = d\n        picked[i] = True\n        taken += 1\n    return picked\n\n\ndef _block_bootstrap(y: np.ndarray, a: np.ndarray, b: np.ndarray, weeks: np.ndarray, reps: int, seed: int) -> dict[str, Any]:\n    r = rt.paired_bootstrap(y, a, b, weeks, reps, seed)\n    return {\"delta_mean\": r[\"mean\"], \"ci95_low\": r[\"ci95_low\"], \"ci95_high\": r[\"ci95_high\"], \"reps\": r[\"reps\"]}\n\n\ndef stratum_metrics(mask: np.ndarray, y_all: np.ndarray, g2: np.ndarray, g2_cal: np.ndarray, r1: np.ndarray,\n                    weeks: np.ndarray, alerts: np.ndarray, cfg: dict[str, Any]) -> dict[str, Any]:\n    from sklearn.metrics import average_precision_score\n\n    known = mask & (y_all >= 0)\n    y = y_all[known]\n    pos = int((y == 1).sum())\n    out: dict[str, Any] = {\"selectable\": int(mask.sum()), \"known\": int(known.sum()), \"positives\": pos,\n                           \"unknown_share\": float(1 - known.sum() / max(mask.sum(), 1)),\n                           \"prevalence\": float(y.mean()) if len(y) else None}\n    ok = len(y) > 0 and 0 < pos < len(y)\n    if ok:\n        pg, pr = float(average_precision_score(y, g2[known])), float(average_precision_score(y, r1[known]))\n        out.update({\"pr_auc_G2\": pg, \"pr_auc_R1\": pr, \"lift_G2\": pg / out[\"prevalence\"], \"lift_R1\": pr / out[\"prevalence\"],\n                    \"brier_G2\": float(np.mean((g2_cal[known] - y) ** 2)), \"ece10_G2\": rt.ece(y, g2_cal[known])})\n        if pos >= cfg[\"min_positives_for_bootstrap\"]:\n            out[\"bootstrap_G2_minus_R1\"] = _block_bootstrap(y, g2[known], r1[known], weeks[known],\n                                                             cfg[\"bootstrap_reps\"], cfg[\"random_seed\"])\n    a = mask & alerts\n    out[\"top_k_alerts\"] = {\"alerts\": int(a.sum()), \"known\": int((a & (y_all >= 0)).sum()),\n                           \"true_positive\": int((a & (y_all == 1)).sum()), \"unknown\": int((a & (y_all < 0)).sum())}\n    return out\n\n\n# ---------------------------------------------------------------------------\n# Прогон\n# ---------------------------------------------------------------------------\ndef run_audit_17(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    import polars as pl\n\n    t0 = time.time()\n    out_dir = Path(cfg[\"output_dir\"])\n    out_dir.mkdir(parents=True, exist_ok=True)\n    parts, manifest = rt.discover_panel(Path(cfg[\"input_dir\"]), cfg[\"verify_sha256\"])\n    manifest_path = sorted(Path(cfg[\"input_dir\"]).rglob(\"panel_manifest_v3.json\"))[0]\n    bdir, binfo = find_bundle(Path(cfg[\"input_dir\"]), out_dir)\n    pred = load_predictor(bdir)\n    bundle = pred.load_bundle(bdir)\n    spec = bundle[\"spec\"]\n    panel_info = check_panel_matches_bundle(manifest, manifest_path, spec)\n    names = [f[\"name\"] for f in spec[\"features\"]]\n    log(f\"панель {manifest['rows']:,} строк; bundle {spec.get('model_version', spec['model_sha256'][:12])}, признаков {len(names)}\")\n    first = min(p[0] for p in PERIODS.values()) - timedelta(days=90)\n    last = max(p[1] for p in PERIODS.values()) + timedelta(days=90)\n    if manifest.get(\"data_end\"):\n        last = min(last, date.fromisoformat(str(manifest[\"data_end\"])[:10]) + timedelta(days=1))\n    activity = object_gas_activity(pl, parts, first, last)\n    runs = silence_runs(pl, activity, cfg[\"min_silence_days\"])\n    windows = {f\"posthoc_after{a}\": posthoc_days(pl, runs, cfg[\"before_days\"], a) for a in AFTER_VARIANTS}\n    controls = {f\"control_shift{s}\": posthoc_days(pl, runs, cfg[\"before_days\"], cfg[\"after_days\"], s) for s in CONTROL_SHIFTS}\n    main_window = windows[f\"posthoc_after{cfg['after_days']}\"]\n    controls = {k: v.join(main_window, on=[\"d_object_key\", \"day\"], how=\"anti\") for k, v in controls.items()}\n    trailing = trailing_days(pl, runs, cfg[\"lookback_days\"])\n    payload: dict[str, Any] = {\n        \"status\": None, \"mode\": cfg[\"mode\"], \"bundle\": {\"model_sha256\": spec[\"model_sha256\"],\n                                                        \"model_version\": spec.get(\"model_version\"), **binfo},\n        \"panel\": panel_info, \"audit_version\": \"17-v2\",\n        \"protocol\": {k: cfg[k] for k in (\"min_silence_days\", \"before_days\", \"after_days\", \"lookback_days\", \"budget_per_day\",\n                                         \"bootstrap_reps\", \"cooldown_days\")},\n        \"note\": (\"ретроспективный audit: 2025 H2 и 2026 H1 уже исследовались; post-hoc окна используют будущее и \"\n                 \"не являются backend-правилом; ничего не выбирается\"),\n        \"silence_runs\": {\"objects\": int(activity[\"d_object_key\"].n_unique()), \"runs\": runs.height,\n                         \"median_length_days\": float(runs[\"length\"].median()) if runs.height else None},\n        \"periods\": {}}\n    for pname in cfg[\"periods\"]:\n        period = PERIODS[pname]\n        frame = load_period(pl, parts, names, cfg, period)\n        if frame.height == 0:\n            payload[\"periods\"][pname] = {\"status\": \"no_rows\"}\n            continue\n        for wname, days in {**windows, **controls}.items():\n            frame = mark(pl, frame, days, wname, 2)          # окно исхода = D + 2\n        frame = mark(pl, frame, trailing, \"trailing_flag_at_D\", 0)\n        records = frame.select([pl.col(\"d_cutoff_date\").cast(pl.String).alias(\"as_of_date\")] + names).to_pandas()\n        res = pred.predict(records, bundle)\n        g2 = np.array([np.nan if r[\"raw_score\"] is None else r[\"raw_score\"] for r in res], dtype=float)\n        g2_cal = np.array([np.nan if r[\"score\"] is None else r[\"score\"] for r in res], dtype=float)\n        eligible = ~np.isnan(g2)\n        r1 = rt.rule_score(frame, rt.TARGET_SPECS[TARGET])\n        y_all = frame[TARGET].fill_null(-1).to_numpy().astype(int)\n        days = np.array([(d - date(2019, 1, 1)).days for d in frame[\"d_cutoff_date\"].to_list()])\n        weeks = days // 7\n        chans = frame[\"d_channel_key\"].to_numpy()\n        g2f = np.nan_to_num(g2, nan=-1.0)\n        # одна и та же скорируемая когорта для обеих политик: abstain не занимает слот тревоги\n        idx = np.flatnonzero(eligible)\n        alerts = np.zeros(frame.height, dtype=bool)\n        alerts_r1 = np.zeros(frame.height, dtype=bool)\n        alerts[idx] = select_alerts(days[idx], chans[idx], g2f[idx], cfg[\"budget_per_day\"], cfg[\"cooldown_days\"])\n        alerts_r1[idx] = select_alerts(days[idx], chans[idx], np.asarray(r1)[idx], cfg[\"budget_per_day\"], cfg[\"cooldown_days\"])\n        clean = (frame[cfg[\"clean_history_col\"]].fill_null(0).to_numpy() == 0) if cfg[\"clean_history_col\"] in frame.columns \\\n            else np.ones(frame.height, dtype=bool)\n        base = eligible\n        strata: dict[str, np.ndarray] = {\"all\": base}\n        for wname in list(windows) + list(controls):\n            m = frame[wname].to_numpy()\n            strata[f\"{wname}:in\"], strata[f\"{wname}:out\"] = base & m, base & ~m\n        tf = frame[\"trailing_flag_at_D\"].to_numpy()\n        strata[\"trailing_flag_at_D:true\"], strata[\"trailing_flag_at_D:false\"] = base & tf, base & ~tf\n        main_out = ~frame[f\"posthoc_after{cfg['after_days']}\"].to_numpy()\n        strata[\"clean30d\"] = base & clean\n        strata[\"clean30d_and_posthoc_out\"] = base & clean & main_out\n        pr: dict[str, Any] = {\"rows_selectable\": frame.height, \"rows_scored\": int(eligible.sum()),\n                              \"alerts_policy\": f\"top-{cfg['budget_per_day']}/сутки, cooldown {cfg['cooldown_days']} сут, \"\n                                               \"G2 и R1 на одной eligible-когорте\",\n                              \"strata\": {}}\n        for sname, m in strata.items():\n            st = stratum_metrics(m, y_all, g2f, np.nan_to_num(g2_cal), r1, weeks, alerts, cfg)\n            st[\"top_k_alerts_R1_policy\"] = {\"alerts\": int((m & alerts_r1).sum()), \"true_positive\": int((m & alerts_r1 & (y_all == 1)).sum()),\n                                            \"unknown\": int((m & alerts_r1 & (y_all < 0)).sum())}\n            pr[\"strata\"][sname] = st\n        # специфичность: обогащение положительных внутри окна по сравнению со сдвинутыми окнами\n        enrich = {}\n        for wname in [f\"posthoc_after{cfg['after_days']}\"] + list(controls):\n            i, o = pr[\"strata\"][f\"{wname}:in\"], pr[\"strata\"][f\"{wname}:out\"]\n            enrich[wname] = (i[\"prevalence\"] / o[\"prevalence\"]) if i.get(\"prevalence\") and o.get(\"prevalence\") else None\n        pr[\"positive_enrichment_in_vs_out\"] = enrich\n        payload[\"periods\"][pname] = pr\n        log(f\"{pname}: selectable {frame.height:,}, скорено {int(eligible.sum()):,}; \"\n            f\"обогащение в окне {enrich.get('posthoc_after' + str(cfg['after_days']))}\")\n    payload[\"status\"] = \"completed_smoke_non_comparable\" if cfg[\"mode\"] == \"SMOKE\" else \"completed\"\n    payload[\"runtime_s\"] = round(time.time() - t0, 1)\n    (out_dir / \"results_17_maintenance_audit.json\").write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=_jd),\n                                                                encoding=\"utf-8\")\n    (out_dir / \"summary_17_maintenance_audit_ru.md\").write_text(summary_markdown(payload, cfg), encoding=\"utf-8\")\n    return payload\n\n\ndef _jd(o: Any) -> Any:\n    if isinstance(o, np.generic):\n        return o.item()\n    if isinstance(o, date):\n        return o.isoformat()\n    raise TypeError(type(o).__name__)\n\n\ndef _f(x: Any, d: int = 3) -> str:\n    return \"—\" if x is None else f\"{x:.{d}f}\"\n\n\ndef summary_markdown(p: dict[str, Any], cfg: dict[str, Any]) -> str:\n    main = f\"posthoc_after{cfg['after_days']}\"\n    L = [f\"# 17 · Audit газового bundle внутри/вне окон сервисных работ — {p['status']}\", \"\",\n         f\"Bundle `{p['bundle'].get('model_version') or p['bundle']['model_sha256'][:12]}` не менялся; обучения нет. {p['note']}.\", \"\",\n         f\"Окно работ: все газовые каналы объекта без показаний ≥ {cfg['min_silence_days']} суток → \"\n         f\"[начало − {cfg['before_days']}; конец + {cfg['after_days']}] (post-hoc). Причинный флаг: молчание закончилось \"\n         f\"в последние {cfg['lookback_days']} суток. Строка в окне, если день исхода D+2 в окне.\", \"\",\n         f\"Версия audit: {p.get('audit_version')}. Панель совпадает с bundle: {p['panel']['panel_matches_bundle']} \"\n         f\"(manifest {p['panel']['manifest_sha256'][:12]}…, строк {p['panel']['rows']:,}); ZIP сверен с .sha256.\", \"\",\n         f\"Найдено отрезков молчания (только между двумя реальными показаниями): {p['silence_runs']['runs']} на \"\n         f\"{p['silence_runs']['objects']} объектах (медиана {_f(p['silence_runs']['median_length_days'], 1)} сут). \"\n         \"Контрольные окна не пересекаются с основным; тревоги G2 и R1 — на одной eligible-когорте.\", \"\"]\n    for pname, pr in p[\"periods\"].items():\n        if \"strata\" not in pr:\n            continue\n        L += [f\"## {pname}\", \"\",\n              \"| страта | selectable | известно | 1 | prevalence | PR-AUC G2 | PR-AUC R1 | lift G2 | lift R1 | ΔPR-AUC G2−R1 [95 %] | top-25: тревог / верных / unknown |\",\n              \"|---|---|---|---|---|---|---|---|---|---|---|\"]\n        show = [\"all\", f\"{main}:in\", f\"{main}:out\", \"posthoc_after14:out\", \"posthoc_after30:out\",\n                \"control_shift30:in\", \"control_shift60:in\", \"trailing_flag_at_D:true\", \"trailing_flag_at_D:false\",\n                \"clean30d\", \"clean30d_and_posthoc_out\"]\n        for s in show:\n            st = pr[\"strata\"].get(s)\n            if not st:\n                continue\n            b = st.get(\"bootstrap_G2_minus_R1\")\n            ci = f\"{_f(b['delta_mean'])} [{_f(b['ci95_low'])}; {_f(b['ci95_high'])}]\" if b and b.get(\"delta_mean\") is not None else \"—\"\n            a = st[\"top_k_alerts\"]\n            L.append(f\"| {s} | {st['selectable']:,} | {st['known']:,} | {st['positives']:,} | {_f(st['prevalence'], 4)} | \"\n                     f\"{_f(st.get('pr_auc_G2'))} | {_f(st.get('pr_auc_R1'))} | {_f(st.get('lift_G2'), 1)} | {_f(st.get('lift_R1'), 1)} | \"\n                     f\"{ci} | {a['alerts']:,} / {a['true_positive']:,} / {a['unknown']:,} |\")\n        e = pr[\"positive_enrichment_in_vs_out\"]\n        L += [\"\", \"Обогащение положительных (prevalence в окне / вне окна): реальные окна \"\n              f\"{_f(e.get(main), 1)}×, сдвинутые на +30 — {_f(e.get('control_shift30'), 1)}×, на +60 — {_f(e.get('control_shift60'), 1)}×.\", \"\"]\n    L += [\"## Как читать\", \"\",\n          \"- Сравнивать lift (PR-AUC / prevalence) и ΔPR-AUC с R1 внутри одной страты, а не голые PR-AUC между стратами.\",\n          \"- Если вне окон G2 ≈ R1 — прогноз реальной газовой опасности не подтверждён; если G2 > R1 с интервалом выше 0 — \"\n          \"сигнал вне сервисных периодов есть (природа не доказана).\",\n          \"- Обогащение в реальных окнах, заметно большее, чем в сдвинутых, — свидетельство связи положительных с работами.\",\n          \"- Ключей каналов/объектов и построчных прогнозов в выходах нет.\", \"\"]\n    return \"\\n\".join(L)\n"
}


In [ ]:
import tempfile
from pathlib import Path
CODE_DIR = Path(tempfile.mkdtemp(prefix="ldt17_code_"))
for name, text in SOURCES.items():
    (CODE_DIR / name).write_text(text, encoding="utf-8")
sys.path.insert(0, str(CODE_DIR))
import maintenance_audit_v3 as ma
print("код записан:", ", ".join(sorted(SOURCES)))


## Прогон (только скоринг)


In [ ]:
CONFIG = ma.make_config_17(MODE)
RESULT = ma.run_audit_17(CONFIG)
print("Статус:", RESULT["status"], "| время:", RESULT["runtime_s"], "с")


## Сводка


In [ ]:
from IPython.display import Markdown, display
OUT = Path(CONFIG["output_dir"])
display(Markdown((OUT / "summary_17_maintenance_audit_ru.md").read_text(encoding="utf-8")))
for name in ("results_17_maintenance_audit.json", "summary_17_maintenance_audit_ru.md"):
    p = OUT / name
    print("-", name, f"({p.stat().st_size / 1024:.0f} КБ)" if p.exists() else "— НЕ СОЗДАН")
